<a href="https://colab.research.google.com/github/GoncalvesAlessandro/pocs-arquiteturas-busca/blob/main/algoritmo_busca_bm25.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
!pip install rank_bm25 -q

import json
import re
import time
from rank_bm25 import BM25Okapi
from google.colab import drive
import os

drive.mount('/content/drive')
caminho_ficheiro = '/content/drive/MyDrive/TCC_Base_Dados_POC/dataset_constituicao.json'

with open(caminho_ficheiro, 'r', encoding='utf-8') as f:
    artigos_cf = json.load(f)

textos_corpus = [doc['texto'] for doc in artigos_cf]

def limpar_texto(texto):
    texto_limpo = re.sub(r'[^\w\s]', '', texto.lower())
    return texto_limpo.split()

print("A construir o índice estatístico do BM25...")

# Medir o Tempo de Indexação (Preparação do motor)
inicio_indexacao = time.perf_counter()

corpus_tokenizado = [limpar_texto(texto) for texto in textos_corpus]
bm25 = BM25Okapi(corpus_tokenizado)

fim_indexacao = time.perf_counter()
tempo_indexacao = fim_indexacao - inicio_indexacao

print(f"Índice BM25 pronto! \n⏱️ Tempo de indexação: {tempo_indexacao:.4f} segundos\n")
print("-----")

# Medir o Tempo de Busca (Latência)
consulta = "quais as regras para a eleição do presidente"
tokens_consulta = limpar_texto(consulta)

inicio_busca = time.perf_counter()

# Obter os 3 resultados com maior pontuação (score)
resultados = bm25.get_top_n(tokens_consulta, textos_corpus, n=3)

fim_busca = time.perf_counter()
tempo_busca = fim_busca - inicio_busca

print(f"Consulta: '{consulta}'")
print(f"Tempo de resposta da busca: {tempo_busca:.6f} segundos\n")

for i, res in enumerate(resultados, 1):
    print(f"🏆 Top {i}:\n{res}\n")

Mounted at /content/drive
A construir o índice estatístico do BM25...
✅ Índice BM25 pronto! 
⏱️ Tempo de indexação: 0.2062 segundos

--------------------------------------------------
🔍 Consulta: 'quais as regras para a eleição do presidente'
⏱️ Tempo de resposta da busca: 0.023222 segundos

🏆 Top 1:
§ 2º A eleição do Governador e do Vice-Governador, observadas as regras do 
	art. 77, e dos Deputados Distritais coincidirá com a dos Governadores e 
	Deputados Estaduais, para mandato de igual duração.

🏆 Top 2:
§ 1º A eleição do Presidente da República importará a do Vice-Presidente com 
	ele registrado.

🏆 Top 3:
c) na hipótese de crédito cedido, regulamento 
disporá sobre as regras às quais se submeterão os sujeitos passivos;(Incluído pela Emenda Constitucional 
	nº 136, de 2025)

